# HADES Paper 2: main sweep
Decision map for Paper 2: for each feedstock class, province and heat supply, how often Pathway A (HTC, then digestion of the process water) has lower impacts than Pathway B (digestion, then HTC of the digestate), and what drives it. Design in the decision log, F13 to F16.

**Grid** (the reader of the guidebook knows or chooses these): feedstock class (4), province (10 provinces and Yukon), heat supply (3), HTC temperature 180 / 220 / 240 °C, residence time 60 / 240 min, HRT 20 / 30 / 40 d for Pathway B only (Pathway A digests soluble process water, HRT fixed at 30 d).
**Sampled** (Latin hypercube, `N_SAMPLES` per cell, A and B paired on the same samples): feedstock composition within its class (`feedstock_sampler`, F16), feed moisture within its class range (F15), and every uncertain parameter with a range (F3, including those the screening found negligible: with a Latin hypercube they cost no extra runs).
**Fixed** (F13): HTC slurry moisture 0.85, Pathway A digester feed 25 kg COD/m³, Pathway B digester feed 10 % TS; ambient temperature from each province's climate normal (F11).
**Not included yet**: HTC surrogate prediction error (F2, after the supervisor meeting). Economics.

Same engine as the Morris notebook (`ADM1/sweep_engine.py`): one ADM1 run per point, scored for every province and heat supply; every point saved to its own file, so the run resumes where it stopped.

Steps: 1 settings · 2 design · 3 unit scores · 4 check against Brightway · 5 pilot · 6 pilot checks · 7 full run · 8 results.

## 1. Settings

In [ ]:
PROJECT = "my_hades_project"                 # Brightway project
EI_NAME = "ecoinvent-3.12-cutoff"
WORKERS = 12                                 # parallel workers (16 cores, 15 GB RAM: 12 leaves headroom)
SWEEP_NAME = "main_v1"                       # output folder results/sweeps/<SWEEP_NAME>

IW_MID = "IMPACT World+ Midpoint 2.2.1_regionalized for ecoinvent v3.12"
IW_DMG = "IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12"
METHODS = {"GWP100 (kg CO2 eq)":            (IW_MID, "Midpoint", "Climate change, short term"),
           "Human health (DALY)":           (IW_DMG, "Human health", "Total human health"),
           "Ecosystem quality (PDF.m2.yr)": (IW_DMG, "Ecosystem quality", "Total ecosystem quality")}

# grid
FEEDSTOCKS = ["food_waste", "ofmsw", "sewage_sludge", "dairy_manure"]
CONTEXTS = ["QC", "ON", "MB", "SK", "AB", "BC", "NB", "NS", "PE", "NL", "YT"]   # NT, NU: below 0 C (F11)
HEAT_CONFIGS = ["paper1", "all_ng", "own_biogas"]
T_LEVELS = [180, 220, 240]                   # C
RT_LEVELS = [60, 240]                        # min

# Latin hypercube
N_SAMPLES = 100                              # per cell; P(A better) known to about +-5 %
SEED = 20261125
N_PILOT_SAMPLES = 2                          # pilot: the first samples of the design (about 190 points)

# Pathway B feed chemistry (decision log C1d): "legacy" (Paper 1 inputs) or "fisgativa"
# (measured inorganic carbon, ammonium, ions and lipid share, Fisgativa et al. 2020).
# Changing it changes the design: use a new SWEEP_NAME.
B_FEED_CHEMISTRY = "legacy"

## 2. Design
Builds every point and stores the design in `design.json`; stops if the output folder holds results of a different design.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "ADM1")) else os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "ADM1"))
import sweep_engine as se                    # workers import it by name, so it must be importable
import feedstock_sampler as fsamp
sb = se._load("system_builder", "system_builder.py")
OUT = os.path.join(ROOT, "results", "sweeps", SWEEP_NAME)
SCORES = os.path.join(OUT, "unit_scores.json")
POINTS_DIR = os.path.join(OUT, "points")
os.makedirs(POINTS_DIR, exist_ok=True)

# feedstocks: compositions from the sampler (one per sample), moisture sampled in the F15 range
corr, n_corr = fsamp.correlations()
feeds = {}
for i, k in enumerate(FEEDSTOCKS):
    wc_lo, wc_hi, wc_c = fsamp.RANGES[k]["WC"]
    comps = fsamp.sample(k, N_SAMPLES, seed=SEED + i, corr=corr[k])
    feeds[k] = dict(feedstock_type=k, feed=dict(WC=float(wc_c)), WC_range=(float(wc_lo), float(wc_hi)),
                    compositions=comps)
    print(f"{k}: {N_SAMPLES} compositions, correlations from "
          + (f"{n_corr[k]} own compositions" if n_corr[k] > 0 else f"{-n_corr[k]} pooled compositions")
          + f"; moisture {wc_lo}-{wc_hi} %")

space = se.parameter_space(sb, sb.hce, sb._adm1_core, include_design=False, include_feed_moisture=True,
                           exclude=("T_AMB_C",))       # ambient: province climate normal (F11)
names = [s["name"] for s in space]
X = se.lhs_design(N_SAMPLES, len(space), seed=SEED)
points = se.make_points(X, feeds, T_LEVELS, RT_LEVELS, ["A", "B"], space, sb._adm1_core, prefix="s",
                        extra={**se.MAIN_SWEEP_FIXED_DESIGN, "b_feed_chemistry": B_FEED_CHEMISTRY},
                        HRT_levels=se.MAIN_SWEEP_HRT_LEVELS)

design = dict(names=names, X=X.tolist(), seed=SEED, feedstocks=FEEDSTOCKS,
              compositions={k: f["compositions"] for k, f in feeds.items()},
              WC_range={k: f["WC_range"] for k, f in feeds.items()},
              contexts=CONTEXTS, heat_configs=HEAT_CONFIGS, T=T_LEVELS, RT=RT_LEVELS,
              fixed=se.MAIN_SWEEP_FIXED_DESIGN, HRT=se.MAIN_SWEEP_HRT_LEVELS, b_feed_chemistry=B_FEED_CHEMISTRY)
design = json.loads(json.dumps(design, default=str))
dpath = os.path.join(OUT, "design.json")
if os.path.exists(dpath) and json.load(open(dpath)) != design and os.listdir(POINTS_DIR):
    raise RuntimeError(f"{OUT} holds results of a different design. Change SWEEP_NAME in step 1 "
                       "(or delete that folder) so old points are not reused.")
json.dump(design, open(dpath, "w"), indent=1)
n_A = sum(p["pathway"] == "A" for p in points)
print(f"\n{len(space)} sampled factors: {names}")
print(f"{len(points)} points ({n_A} A, {len(points) - n_A} B), each scored for {len(CONTEXTS)} provinces "
      f"x {len(HEAT_CONFIGS)} heat supplies = {len(points) * len(CONTEXTS) * len(HEAT_CONFIGS):,} evaluations")

## 3. Unit scores (once per design, serial)
Probe points at the central composition and the centre of every range, at the extreme temperatures and every residence time and HRT, collect the background datasets; their unit scores are stored in `unit_scores.json`. Re-run this cell if step 6 or 8 reports missing keys.

In [ ]:
import bw2data as bd
probe_feeds = {k: dict(f, compositions=[fsamp.central(k)]) for k, f in feeds.items()}
probes = se.make_points(np.full((1, len(space)), 0.5), probe_feeds, [min(T_LEVELS), max(T_LEVELS)], RT_LEVELS,
                        ["A", "B"], space, sb._adm1_core, prefix="probe",
                        extra={**se.MAIN_SWEEP_FIXED_DESIGN, "b_feed_chemistry": B_FEED_CHEMISTRY},
                        HRT_levels={"A": [30], "B": [30]})
scores = se.precompute_scores(probes, HEAT_CONFIGS, CONTEXTS, list(METHODS.values()), PROJECT, EI_NAME, SCORES)

## 4. Check the engine against a full Brightway LCA
Two probe points rebuilt in a real foreground database and computed with bw2calc; stops if any impact differs by more than 0.01 %.

In [ ]:
for pt, ctx_, heat in [(probes[0], "AB", "own_biogas"), (probes[-1], "NS", "all_ng")]:
    chk = se.validate_point(pt, ctx_, heat, METHODS, PROJECT, EI_NAME, SCORES)
    print(pt["point_id"], ctx_, heat); display(chk)

## 5. Pilot
The first `N_PILOT_SAMPLES` samples of the design, for every feedstock, T, RT, pathway and HRT (about 190 points). They are part of the main design, so the full run in step 7 skips them.

In [ ]:
import time
LABELS = se.method_labels_for(METHODS)
pilot = [p for p in points if p["sample_id"] < N_PILOT_SAMPLES]
t0 = time.time()
se.run_sweep(pilot, HEAT_CONFIGS, CONTEXTS, LABELS, PROJECT, EI_NAME, SCORES, POINTS_DIR, workers=WORKERS)
dt = time.time() - t0
print(f"{len(pilot)} points in {dt / 60:.1f} min -> full design about {dt * len(points) / len(pilot) / 3600:.1f} h")

## 6. Pilot checks
Failed evaluations, missing unit scores, binding caps, surrogate domain, digester ammonia (Pathway A at 25 kg COD/m³; inhibition reported from 1.7 to 14 g N/L, Chen, Cheng & Creamer 2008, Bioresour. Technol. 99:4044) and pH. Look at these before step 7.

In [ ]:
def load(out_dir=POINTS_DIR):
    df = se.collect(out_dir)
    if "error" in df:
        err = df[df["error"].notna()]
        print(f"{len(err)} failed evaluations")
        if len(err):
            display(err[["point_id", "context", "heat_config", "error"]].head(20))
        df = df[df["error"].isna()]
    miss = df["missing_keys"].fillna("")
    print(f"{(miss != '').sum()} evaluations with missing unit scores (re-run step 3)")
    return df

pdf = load()
one = pdf.drop_duplicates("point_id")            # process results do not depend on province or heat supply
print(f"{one['point_id'].nunique()} points, {len(pdf)} evaluations")
caps = one.assign(caps=one["caps_active"].fillna("")).query("caps != ''")
print(f"points with a binding cap: {len(caps)} of {len(one)}")
if len(caps):
    display(caps.groupby(["feedstock", "pathway"])["caps"].agg(lambda s: "; ".join(sorted(set("; ".join(s).split("; ")))[:6])))
flags = one["qa_flags"].fillna("")
print("surrogate outside joint domain:", flags.str.contains("outside joint domain").sum())
display(one.groupby(["feedstock", "pathway"])[["_TAN_gN_L", "_NH3_free_gN_L", "_pH"]].describe()
           .loc[:, (slice(None), ["min", "50%", "max"])].round(2))

## 7. Full run
Resumes automatically: points already in `points/` (the pilot included) are skipped. About 9,600 points; can run over several sessions.

In [ ]:
se.run_sweep(points, HEAT_CONFIGS, CONTEXTS, LABELS, PROJECT, EI_NAME, SCORES, POINTS_DIR, workers=WORKERS)

## 8. Results: how often A beats B
Each B point is paired with the A point of the same sample, feedstock, T and RT (A has one HRT, B three). Share of samples where A has the lower impact, per feedstock, province and heat supply; then by T, RT and HRT. Indicative until the surrogate error is sampled (F2).

In [ ]:
df = load()
KEYS = ["feedstock", "T", "RT", "context", "heat_config", "sample_id"]
pairs = {}
for lab in METHODS:
    a = df[df["pathway"] == "A"].set_index(KEYS)[lab].rename("A")
    b = df[df["pathway"] == "B"].set_index(KEYS)[[lab, "HRT"]].rename(columns={lab: "B"})
    w = b.join(a, how="inner").reset_index()
    w["A_better"] = w["A"] < w["B"]
    w["A_minus_B"] = w["A"] - w["B"]
    pairs[lab] = w
    print(f"\n{lab}: {len(w):,} pairs. Share of samples where A beats B (all T, RT, HRT)")
    display(w.pivot_table(index=["feedstock", "heat_config"], columns="context", values="A_better", aggfunc="mean")
             .reindex(columns=CONTEXTS).style.format("{:.0%}").background_gradient(cmap="RdBu_r", vmin=0, vmax=1))
lab = list(METHODS)[0]
for by in ("T", "RT", "HRT"):
    print(f"\n{lab}: share A beats B by {by}")
    display(pairs[lab].pivot_table(index="feedstock", columns=by, values="A_better", aggfunc="mean").style.format("{:.0%}"))
pd.concat([w.assign(method=k) for k, w in pairs.items()]).to_csv(os.path.join(OUT, "pairs.csv"), index=False)